In [3]:
import pandas as pd
df = pd.read_excel("../data/raw/Telco_customer_churn.xlsx")
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [4]:
df["Total Charges"] = pd.to_numeric(
    df["Total Charges"],
    errors="coerce",
)
df["Total Charges"] = df["Total Charges"].fillna(0)
# df["Total Charges"] 


In [5]:
target = 'Churn Value'
columns_to_exclude = ['CustomerID', 'Count', 'Country',
'State', 'Churn Label', 'Churn Score', 'Churn Reason', 'City','Zip Code', 'Lat Long', 'Latitude',
'Longitude', 'CLTV']

In [7]:
X =df.drop(columns=[target] + columns_to_exclude)
y = df[target]

In [8]:
X.head()

,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50
3,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05
4,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30


In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y # contains same proportion of classes as the original dataset in both the training and testing sets
)

In [10]:
print("X_train", X_train.shape)
print("X_test", X_test.shape)
print("y_train", y_train.shape)
print("y_test", y_test.shape)

X_train (5634, 19)
X_test (1409, 19)
y_train (5634,)
y_test (1409,)


In [11]:
print("Training distribution:")
print(y_train.value_counts(normalize=True))

print("Testing distribution:")
print(y_test.value_counts(normalize=True))

Training distribution:
Churn Value
0    0.734647
1    0.265353
Name: proportion, dtype: float64
Testing distribution:
Churn Value
0    0.734564
1    0.265436
Name: proportion, dtype: float64


In [12]:
#Select Numerical columns
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

#select categorical columns
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

Numerical columns: ['Tenure Months', 'Monthly Charges', 'Total Charges']
Categorical columns: ['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method']


C:\Users\Khan\AppData\Local\Temp\ipykernel_11124\1584273764.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

#1. build the traffic Cop (ColumnTransformer)
preprocessor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first'), cat_cols)
    ]
)

#2. Fit on the training data, then transform the training data
X_train_processed = preprocessor.fit_transform(X_train)

#3. Only transform the testing data (do not fit test data)
X_test_processed = preprocessor.transform(X_test)

#4. New Shapes
print("Original X_train shape:", X_train.shape)
print("Processed X_train shape:", X_train_processed.shape)


Original X_train shape: (5634, 19)
Processed X_train shape: (5634, 30)


In [17]:
from sklearn.dummy import DummyClassifier, DummyClassifier

#1. Create a dummy classifier
dummy_clf = DummyClassifier(strategy="most_frequent")

#2. "Train" it on the processed training data
dummy_clf.fit(X_train_processed, y_train)

#3. Calculate its accuracy on the  unseen test data
dummmy_accuracy = dummy_clf.score(X_test_processed, y_test)

#4. print accuracy
print("Dummy Classifier Accuracy:", dummmy_accuracy)

Dummy Classifier Accuracy: 0.7345635202271115


In [19]:
#Logistic Regression
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000, random_state=42)

log_reg.fit(X_train_processed, y_train)

log_reg_accuracy = log_reg.score(X_test_processed, y_test)

print("Logistic Regression Accuracy:", log_reg_accuracy)

Logistic Regression Accuracy: 0.8019872249822569


In [22]:
from sklearn.metrics import confusion_matrix, classification_report

#1. get the models actual predictions for the test set
y_pred = log_reg.predict(X_test_processed)

#2. generate and print the confusion matrix
print("Confusion Matrix: ")
print(confusion_matrix(y_test, y_pred))

#3. generate and print the classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix: 
[[916 119]
 [160 214]]

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.64      0.57      0.61       374

    accuracy                           0.80      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.80      0.80      1409



In [23]:
# Get the raw probabilities instead of the final 0 or 1 predictions
y_pred_proba = log_reg.predict_proba(X_test_processed)

# Print the probabilities for the first 5 customers
print("Probabilities for the first 5 customers in X_test:\n")
print(y_pred_proba[:5])

Probabilities for the first 5 customers in X_test:

[[0.92404538 0.07595462]
 [0.32803038 0.67196962]
 [0.89735648 0.10264352]
 [0.52796085 0.47203915]
 [0.96892694 0.03107306]]


In [24]:
from sklearn.metrics import roc_auc_score

#1. Slice the array to get ONLY the probabilities for class 1 (Churn)
y_pred_proba_churn = y_pred_proba[:,1]

#2. calculate the ROC-AUC score
roc_auc = roc_auc_score(y_test, y_pred_proba_churn)

print("ROC-AUC Score:", roc_auc)

ROC-AUC Score: 0.8490531917641891


In [25]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

# 1. Initialize the Random Forest
# We use 100 trees (n_estimators) by default. random_state ensures reproducibility.
rf_clf = RandomForestClassifier(random_state=42)

# 2. Train the model on the exact same processed data
rf_clf.fit(X_train_processed, y_train)

# 3. Get both the hard predictions (0/1) and the raw probabilities
rf_pred = rf_clf.predict(X_test_processed)
rf_pred_proba = rf_clf.predict_proba(X_test_processed)[:, 1]

# 4. Print the evaluation metrics to compare with Logistic Regression
print(f"Random Forest Accuracy: {rf_clf.score(X_test_processed, y_test) * 100:.2f}%\n")

print("Confusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

print(f"\nROC-AUC Score: {roc_auc_score(y_test, rf_pred_proba):.4f}")

Random Forest Accuracy: 79.21%

Confusion Matrix:
[[920 115]
 [178 196]]

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.89      0.86      1035
           1       0.63      0.52      0.57       374

    accuracy                           0.79      1409
   macro avg       0.73      0.71      0.72      1409
weighted avg       0.78      0.79      0.79      1409


ROC-AUC Score: 0.8321


In [26]:
# 1. Initialize a new Logistic Regression model, but add class_weight='balanced'
log_reg_balanced = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)

# 2. Train the model
log_reg_balanced.fit(X_train_processed, y_train)

# 3. Get the new predictions
y_pred_balanced = log_reg_balanced.predict(X_test_processed)

# 4. Print the new Confusion Matrix and Classification Report
print("Confusion Matrix (Balanced):")
print(confusion_matrix(y_test, y_pred_balanced))

print("\nClassification Report (Balanced):")
print(classification_report(y_test, y_pred_balanced))

Confusion Matrix (Balanced):
[[758 277]
 [ 83 291]]

Classification Report (Balanced):
              precision    recall  f1-score   support

           0       0.90      0.73      0.81      1035
           1       0.51      0.78      0.62       374

    accuracy                           0.74      1409
   macro avg       0.71      0.76      0.71      1409
weighted avg       0.80      0.74      0.76      1409



In [27]:
import pandas as pd
import numpy as np

# 1. Get the names of all the features from our ColumnTransformer
feature_names = preprocessor.get_feature_names_out()

# 2. Get the mathematical weights (coefficients) from our balanced model
coefficients = log_reg_balanced.coef_[0]

# 3. Put them in a DataFrame so we can read them easily
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': coefficients
})

# 4. Sort by the absolute value (magnitude) to see the strongest drivers
feature_importance['Abs_Importance'] = np.abs(feature_importance['Importance'])
feature_importance = feature_importance.sort_values(by='Abs_Importance', ascending=False)

# 5. Display the Top 10 most impactful features
print("Top 10 Drivers of Customer Churn:\n")
print(feature_importance[['Feature', 'Importance']].head(10))


Top 10 Drivers of Customer Churn:

                                 Feature  Importance
6                    cat__Dependents_Yes   -1.622896
25                cat__Contract_Two year   -1.379745
0                     num__Tenure Months   -1.216790
10     cat__Internet Service_Fiber optic    1.035607
24                cat__Contract_One year   -0.773395
2                     num__Total Charges    0.516367
1                   num__Monthly Charges   -0.425689
28  cat__Payment Method_Electronic check    0.399711
23             cat__Streaming Movies_Yes    0.365462
21                 cat__Streaming TV_Yes    0.345188


In [28]:
from sklearn.pipeline import Pipeline
import joblib

# 1. Glue the preprocessor and the balanced model together
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor), # Your ColumnTransformer from earlier
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])
final_pipeline.fit(X_train, y_train)
pipeline_accuracy = final_pipeline.score(X_test, y_test)
print(f"Final Pipeline Accuracy: {pipeline_accuracy * 100:.2f}%")

# 4. Save the pipeline to your hard drive for Phase G (Deployment)
joblib.dump(final_pipeline, '../models/churn_prediction_pipeline.pkl')
print("\nSuccess! Pipeline saved to the models folder.")

Final Pipeline Accuracy: 74.45%

Success! Pipeline saved to the models folder.
